# Claim Ingestion (landing -> bronze)

1. Lists the files currently waiting in `landing/ClaimData/`. **No files -> the task ends successfully** instead of failing.
2. Reads **exactly those files** with a fixed schema and appends them to `bronze.claim` with
   `merge_flag = false`, plus `source_file` and `ingestion_timestamp` for traceability.
3. Moves **exactly those files** to `processed/ClaimData/<yyyy-MM-dd>/<HHmmss>/`. A file ADF lands while this
   runs stays in landing for the next run instead of being archived unread.

If a run fails after step 2 but before step 3, the retry appends the same rows again. That is safe: silver
deduplicates on the business key.

**Job order:** before `Silver_Claim_Implementation`.

In [0]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

LANDING   = "abfss://landing@insurancepolicy1adls.dfs.core.windows.net/ClaimData/"
PROCESSED = "abfss://processed@insurancepolicy1adls.dfs.core.windows.net/ClaimData/"
TABLE     = "policysystemdatabricks.bronze.claim"
EXTENSION = '.parquet'  # None = take every data file in the folder
SCHEMA    = "claim_id INT, policy_id INT, date_of_claim TIMESTAMP, claim_amount DOUBLE, claim_status STRING, LastUpdatedTimeStamp TIMESTAMP"

In [0]:
def landing_files(path):
    """Data files in the landing folder; [] when the folder is empty or does not exist."""
    try:
        entries = dbutils.fs.ls(path)
    except Exception as e:
        msg = str(e).lower()
        if "not found" in msg or "notfound" in msg or "does not exist" in msg:
            return []
        raise  # anything else (e.g. permissions) must still fail the task
    return [f for f in entries
            if not f.name.endswith("/")                 # skip sub-folders
            and not f.name.startswith(("_", "."))       # skip _SUCCESS, .crc and similar
            and (EXTENSION is None or f.name.lower().endswith(EXTENSION))]

files = landing_files(LANDING)
print(f"{len(files)} file(s) waiting in {LANDING}")
for f in files:
    print(" -", f.name, f.size, "bytes")

if not files:
    dbutils.notebook.exit(f"No new files in {LANDING} - nothing to ingest.")

In [0]:
paths = [f.path for f in files]

df = (spark.read.schema(SCHEMA).parquet(*paths)
      .withColumn("merge_flag", F.lit(False))
      .withColumn("source_file", F.col("_metadata.file_path"))
      .withColumn("ingestion_timestamp", F.current_timestamp()))

# mergeSchema adds source_file / ingestion_timestamp to the existing table on the first run.
df.write.mode("append").option("mergeSchema", "true").saveAsTable(TABLE)

# Row count from the Delta log of the write that just happened (no second read of the files)
last_write = spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 1").first()
print(f"{last_write['operation']}: {last_write['operationMetrics'].get('numOutputRows')} row(s) appended to {TABLE}")

WRITE: 0 row(s) appended to policysystemdatabricks.bronze.claim


In [0]:
run_folder = datetime.now(timezone.utc).strftime("%Y-%m-%d/%H%M%S")  # sortable, unique per run (UTC)
target = f"{PROCESSED}{run_folder}/"

for f in files:
    dbutils.fs.mv(f.path, target + f.name)

print(f"Moved {len(files)} file(s) to {target}")

Moved 0 file(s) to abfss://processed@insurancepolicy1adls.dfs.core.windows.net/ClaimData/2026-11-02/071657/
